# Example 3.4 / 1: Print a Short Report

Companion notebook for **3.4 "print and printf"** in the AMPL lab Documentation, section "Print a Short Report". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Print a Short Report

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                # Products
param profit {PROD};     # Profit per ton
param Plan {PROD} >= 0;  # Planned tons, given as data (no solve)

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.param["profit"] = {"bands": 25, "coils": 30}
ampl.param["Plan"] = {"bands": 6000, "coils": 4000}

In [5]:
# run.py
ampl.eval('print "Total profit:", sum {p in PROD} profit[p] * Plan[p];')
ampl.eval('printf {p in PROD} "%-8s %8.1f\\n", p, Plan[p];')

Total profit: 270000
bands      6000.0
coils      4000.0


### Pure AMPL: data.dat and model.run

Pure AMPL:

In [6]:
%%writefile data.dat
set PROD := bands coils;
param: profit Plan :=
  bands 25 6000
  coils 30 4000;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
print "Total profit:", sum {p in PROD} profit[p] * Plan[p];
printf {p in PROD} "%-8s %8.1f\n", p, Plan[p];

Total profit: 270000
bands      6000.0
coils      4000.0
